# Sesi 1 — Dasar Python dan Pengolahan Citra Radiologi Dentomaksilofasial

**Lab skills PPDGS Radiologi Kedokteran Gigi · 150 menit · tingkat pemula**

Notebook ini memakai empat radiograf panoramik pediatrik yang sudah dipublikasikan. Pada sesi ini kita memakai `test_cate1_001` sebagai kasus utama dan `test_cate1_012` sebagai latihan berpasangan.

> **Batas penggunaan:** materi ini untuk pembelajaran dan audit teknis, bukan perangkat klinis. Jangan memakai output notebook untuk keputusan perawatan pasien.

## Tujuan belajar dan tanda keberhasilan

Setelah sesi ini, Anda diharapkan mampu:

1. menjalankan sel Colab berurutan dan membaca pesan error sederhana;
2. mengenali assignment, `list`, indeks mulai nol, pemanggilan fungsi, `shape`, dan `dtype`;
3. menghubungkan koordinat kotak `[x1, y1, x2, y2]` dengan slicing array `[y1:y2, x1:x2]`;
4. membandingkan citra asli, histogram equalization, dan CLAHE secara kritis; serta
5. menjelaskan mengapa peningkatan kontras tidak menciptakan informasi diagnostik baru.

**Tanda berhasil:** Anda dapat mengganti satu kasus, menentukan ROI yang tidak kosong, dan menjelaskan satu manfaat serta satu risiko CLAHE.

## Agenda 150 menit

| Menit | Kegiatan |
|---:|---|
| 0–15 | Orientasi, Colab, keamanan data |
| 15–45 | Variabel, `list`, indeks, fungsi, dan `print()` |
| 45–75 | OPG sebagai array: `shape`, `dtype`, piksel, histogram |
| 75–90 | Istirahat dan checkpoint |
| 90–115 | Bounding box, koordinat, ROI, cropping |
| 115–135 | Histogram equalization dan CLAHE |
| 135–150 | Tantangan kasus kedua dan exit ticket |

## Sebelum mulai: keamanan data

- **Jangan unggah radiograf pasien, nama, nomor rekam medis, tanggal lahir, atau metadata DICOM ke Colab.**
- Gunakan hanya case ID publik yang sudah disediakan notebook ini.
- Anonimisasi pada suatu dataset publik tidak otomatis membuat data klinis lokal boleh dipindahkan ke cloud.
- Jika ragu terhadap kebijakan institusi, berhenti dan gunakan hanya aset latihan lokal.

Dataset latihan: [Children's Dental Panoramic Radiographs Dataset](https://doi.org/10.6084/m9.figshare.c.6317013.v1), dengan konteks pengumpulan, anonimisasi, dan keterbatasan pada [artikel Scientific Data](https://doi.org/10.1038/s41597-023-02237-5). Rekaman dataset mencantumkan CC0; artikel diterbitkan dengan CC BY 4.0. Provenance dan checksum tiap citra juga tersimpan di manifest aset.

## Cara memakai Colab dan memulihkan error

1. Buka notebook di Colab, lalu pilih **Runtime → Restart session**.
2. Pilih **Runtime → Run all**. Tunggu sampai tanda putar pada sel selesai.
3. Sel berlabel **▶ Jalankan** cukup dijalankan. Sel **✏ Ubah** berisi hanya 1–2 parameter aman untuk dicoba.
4. Jika muncul `NameError`, biasanya sel sebelumnya belum dijalankan: pilih **Run all** lagi.
5. Jika unduhan gagal, periksa koneksi lalu jalankan ulang sel kasus. Notebook tidak akan memakai berkas dengan checksum salah.
6. Bila koneksi kelas putus, gunakan salinan aset publik dan [`canonical_sesi1_overview.png`](../assets/pediatric_opg/canonical_sesi1_overview.png) yang dibagikan pengajar. Visual itu adalah **output tersimpan, bukan eksekusi baru**—jangan menggantinya dengan data pasien.

**Preflight pengajar:** buka notebook pada sesi Colab CPU baru, jalankan semua sel, lalu simpan empat citra publik di cache lokal/USB kelas.

### ▶ Jalankan — setup lingkungan

Sel ini hanya memeriksa pustaka umum yang sudah tersedia di Colab. Tidak diperlukan GPU.

In [ ]:
import hashlib, importlib.util, json, tempfile
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from PIL import Image

print("Python siap")
print("NumPy:", np.__version__, "| OpenCV:", cv2.__version__)
print("Lingkungan:", "Google Colab" if importlib.util.find_spec("google.colab") else "lokal")

## 1. Python seperlunya: objek → fungsi → keluaran

Python membaca sel dari atas ke bawah.

- `=` menyimpan nilai ke sebuah nama (*assignment*);
- `[]` membuat `list` dan juga memilih elemen;
- indeks Python dimulai dari **0**, bukan 1;
- `fungsi(argumen)` menjalankan suatu fungsi; dan
- `print()` membantu kita memeriksa keadaan program.

Baca dulu kode di bawah, prediksi keluarannya, lalu jalankan.

### ▶ Jalankan — assignment dan `print()`

In [ ]:
modalitas = "radiograf panoramik"
jumlah_kasus = 4
ukuran_piksel = "8-bit"

print("Modalitas:", modalitas)
print("Jumlah kasus publik:", jumlah_kasus)
print("Format latihan:", ukuran_piksel)

### ▶ Jalankan — `list` dan indeks mulai nol

In [ ]:
case_ids = ["test_cate1_001", "test_cate1_004", "test_cate1_012", "test_cate1_000"]

print("Elemen indeks 0:", case_ids[0])
print("Elemen indeks 2:", case_ids[2])
print("Jumlah elemen:", len(case_ids))

### ▶ Jalankan — membuat dan memanggil fungsi

Baris yang menjorok ke kanan berada di dalam fungsi. Fungsi membantu menyembunyikan langkah teknis yang panjang.

In [ ]:
def perkenalkan_kasus(case_id, modalitas):
    pesan = f"{case_id} adalah contoh {modalitas} publik."
    return pesan

ringkasan = perkenalkan_kasus(case_ids[0], modalitas)
print(ringkasan)

### 🩺 Diskusikan — error adalah petunjuk

| Pesan | Arti yang paling sering | Tindakan pertama |
|---|---|---|
| `NameError` | nama belum dibuat | jalankan sel dari atas |
| `IndexError` | indeks melewati panjang `list` | cek `len(...)` dan ingat indeks mulai nol |
| `FileNotFoundError` | aset belum ada | periksa koneksi/cache, lalu jalankan ulang sel kasus |
| `RuntimeError: checksum` | isi berkas berbeda | jangan lanjutkan; unduh salinan publik yang benar |

Prediksi: apa yang terjadi bila `case_ids[4]` dipanggil? Jangan jalankan pada saat demonstrasi *Run all*; jawab dengan kata-kata.

### ✅ Checkpoint 1

Tanpa melihat catatan, tunjuk satu contoh:

- nama variabel;
- `list`;
- indeks pertama; dan
- pemanggilan fungsi.

Bila empat istilah ini masih terasa asing, ulangi tiga sel kode terakhir sebelum lanjut.

## 2. Memuat kasus publik dengan pemeriksaan integritas

### ▶ Jalankan — helper kelas

Sel panjang berikut adalah “mesin” notebook. Anda tidak perlu menghafalnya. Helper akan:

- mencari aset repo terlebih dahulu;
- bila perlu, mengunduh aset publik ke folder sementara;
- mencocokkan SHA-256 citra dengan manifest; dan
- berhenti dengan pesan pemulihan jika berkas rusak atau unduhan gagal.

Tekan ikon panah kecil di kiri sel untuk menyembunyikan/menampilkan kode.

In [ ]:
RAW_BASE = (
    "https://raw.githubusercontent.com/kristonova/"
    "introduction-AI-coding-for-radiologist/main/assets/pediatric_opg"
)
CACHE_DIR = Path(tempfile.gettempdir()) / "dental_ai_lab_public_assets"


def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _asset_roots():
    roots = []
    here = Path.cwd().resolve()
    for base in [here, *list(here.parents)[:3]]:
        candidate = base / "assets" / "pediatric_opg"
        if candidate not in roots:
            roots.append(candidate)
    roots.append(CACHE_DIR)
    return roots


def _download(url, destination, timeout=30):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    request = Request(url, headers={"User-Agent": "UGM-dental-AI-lab/1.0"})
    try:
        with urlopen(request, timeout=timeout) as response:
            temporary.write_bytes(response.read())
        temporary.replace(destination)
    except (HTTPError, URLError, TimeoutError, OSError) as exc:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            "Unduhan aset publik gagal. Periksa koneksi, lalu jalankan ulang sel. "
            "Jika kelas sedang offline, gunakan cache publik dari pengajar. "
            "Jangan mengunggah data pasien sebagai pengganti."
        ) from exc
    return destination


def _load_manifest():
    for root in _asset_roots():
        path = root / "cases.json"
        if path.exists():
            try:
                return json.loads(path.read_text(encoding="utf-8"))
            except (json.JSONDecodeError, OSError) as exc:
                raise RuntimeError(
                    f"Manifest tidak dapat dibaca: {path}. "
                    "Ambil ulang manifest publik dari repo."
                ) from exc
    path = _download(f"{RAW_BASE}/cases.json", CACHE_DIR / "cases.json")
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        path.unlink(missing_ok=True)
        raise RuntimeError(
            "Manifest hasil unduhan tidak valid dan sudah dihapus. Jalankan ulang sel."
        ) from exc


def _manifest_cases(manifest):
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict) and isinstance(manifest.get("cases"), list):
        return manifest["cases"]
    raise RuntimeError("Format manifest tidak dikenali: daftar 'cases' tidak ditemukan.")


def _normalise_case_id(case_id):
    case_id = str(case_id).strip()
    return case_id if case_id.startswith("test_") else f"test_{case_id}"


def _find_record(case_id):
    wanted = _normalise_case_id(case_id)
    for record in _manifest_cases(_load_manifest()):
        candidate = _normalise_case_id(record.get("case_id", ""))
        if candidate == wanted:
            return record
    available = [
        _normalise_case_id(item.get("case_id", ""))
        for item in _manifest_cases(_load_manifest())
    ]
    raise ValueError(f"CASE_ID tidak ditemukan. Pilih salah satu: {available}")


def _record_filename(record):
    filename = (
        record.get("image")
        or record.get("image_filename")
        or record.get("filename")
    )
    if not filename:
        raise RuntimeError("Manifest tidak memuat nama berkas citra.")
    return Path(filename).name


def _record_sha(record):
    expected = record.get("sha256") or record.get("image_sha256")
    if not expected:
        raise RuntimeError("Manifest tidak memuat SHA-256 citra; aset tidak digunakan.")
    return str(expected).lower()


def _verified_image_path(record):
    filename = _record_filename(record)
    expected = _record_sha(record)
    mismatches = []
    for root in _asset_roots():
        path = root / filename
        if path.exists():
            actual = _sha256(path)
            if actual == expected:
                return path
            mismatches.append(f"{path} ({actual[:12]}...)")
            if root == CACHE_DIR:
                path.unlink(missing_ok=True)

    download_path = CACHE_DIR / filename
    try:
        _download(f"{RAW_BASE}/{filename}", download_path)
        actual = _sha256(download_path)
        if actual != expected:
            download_path.unlink(missing_ok=True)
            raise RuntimeError(
                "Checksum citra hasil unduhan tidak cocok. Berkas sudah dihapus; "
                "notebook tidak melanjutkan dengan salinan lama."
            )
        return download_path
    except RuntimeError as exc:
        detail = f" Salinan lokal yang ditolak: {mismatches}." if mismatches else ""
        raise RuntimeError(f"{exc}{detail}") from exc


def _annotations(record):
    items = record.get("annotations") or record.get("labels") or []
    normalised = []
    for item in items:
        box = item.get("bbox_xyxy")
        if not isinstance(box, list) or len(box) != 4:
            continue
        normalised.append(
            {
                "bbox_xyxy": [float(value) for value in box],
                "label_id": (
                    item.get("label_indonesia")
                    or item.get("label_idn")
                    or item.get("label_id")
                    or "Label publik"
                ),
                "label_en": (
                    item.get("label_en")
                    or item.get("label_english")
                    or item.get("label_inggris")
                    or ""
                ),
            }
        )
    return normalised


def load_case(case_id):
    """Muat satu kasus publik terverifikasi sebagai array RGB uint8."""
    record = _find_record(case_id)
    path = _verified_image_path(record)
    try:
        image = np.asarray(Image.open(path).convert("RGB"))
    except (OSError, ValueError) as exc:
        if path.parent == CACHE_DIR:
            path.unlink(missing_ok=True)
        raise RuntimeError(
            "Citra tidak dapat dibaca. Salinan cache yang rusak tidak digunakan."
        ) from exc

    if image.ndim != 3 or image.shape[2] != 3 or image.dtype != np.uint8:
        raise RuntimeError("Format aset tidak sesuai: diharapkan PNG RGB 8-bit.")
    return {
        "case_id": _normalise_case_id(record.get("case_id", case_id)),
        "image": image,
        "annotations": _annotations(record),
        "record": record,
        "path": str(path),
    }


def _to_gray(image):
    array = np.asarray(image)
    if array.ndim == 2:
        return array.astype(np.uint8)
    if array.ndim == 3 and array.shape[2] == 3:
        return cv2.cvtColor(array.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    raise ValueError("Citra harus berupa array grayscale atau RGB.")


def _clip_roi(roi, width, height):
    if not isinstance(roi, (list, tuple)) or len(roi) != 4:
        raise ValueError("ROI harus berupa [x1, y1, x2, y2].")
    x1, y1, x2, y2 = [int(round(value)) for value in roi]
    x1, x2 = np.clip([x1, x2], 0, width)
    y1, y2 = np.clip([y1, y2], 0, height)
    if x2 <= x1 or y2 <= y1:
        raise ValueError(
            "ROI kosong setelah dibatasi ke ukuran citra. "
            "Pastikan x2 > x1 dan y2 > y1."
        )
    return int(x1), int(y1), int(x2), int(y2)


def _draw_annotations(axis, annotations):
    for annotation in annotations:
        x1, y1, x2, y2 = annotation["bbox_xyxy"]
        axis.add_patch(
            Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor="#ff4d4d", linewidth=1.8
            )
        )
        axis.text(
            x1, max(0, y1 - 8), annotation["label_id"],
            color="white", fontsize=7,
            bbox={"facecolor": "#b30000", "alpha": 0.75, "pad": 1.5},
        )


def show_roi(image, roi, annotations=None):
    """Tampilkan citra, anotasi, ROI, dan matriks piksel berdampingan."""
    array = np.asarray(image)
    height, width = array.shape[:2]
    x1, y1, x2, y2 = _clip_roi(roi, width, height)
    crop = array[y1:y2, x1:x2]
    gray_crop = _to_gray(crop)

    cy, cx = gray_crop.shape[0] // 2, gray_crop.shape[1] // 2
    py1, px1 = max(0, cy - 4), max(0, cx - 4)
    patch = gray_crop[py1:py1 + 8, px1:px1 + 8]

    figure, axes = plt.subplots(1, 4, figsize=(21, 5))
    axes[0].imshow(array)
    axes[0].set_title("1 · Citra utuh")
    axes[1].imshow(array)
    _draw_annotations(axes[1], annotations or [])
    axes[1].add_patch(
        Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor="#ffd400", linewidth=2.2
        )
    )
    axes[1].set_title("2 · Anotasi merah + ROI kuning")
    axes[2].imshow(crop)
    axes[2].set_title(f"3 · Crop [{y1}:{y2}, {x1}:{x2}]")
    axes[3].imshow(patch, cmap="gray", vmin=0, vmax=255)
    axes[3].set_title(f"4 · Piksel {patch.shape[0]}×{patch.shape[1]}")

    for row in range(patch.shape[0]):
        for column in range(patch.shape[1]):
            value = int(patch[row, column])
            axes[3].text(
                column, row, value, ha="center", va="center",
                fontsize=7, color="black" if value > 145 else "white"
            )
    for axis in axes[:3]:
        axis.axis("off")
    axes[3].set_xticks([])
    axes[3].set_yticks([])
    plt.tight_layout()
    plt.show()
    return crop


def apply_clahe(image, clip_limit=2.0):
    """Terapkan CLAHE pada citra grayscale 8-bit."""
    if not 0.1 <= float(clip_limit) <= 10.0:
        raise ValueError("CLAHE_CLIP harus berada antara 0.1 dan 10.0.")
    gray = _to_gray(image)
    operator = cv2.createCLAHE(
        clipLimit=float(clip_limit), tileGridSize=(8, 8)
    )
    return operator.apply(gray)


def _show_image_and_histogram(images, titles):
    figure, axes = plt.subplots(2, len(images), figsize=(17, 8))
    for column, (image, title) in enumerate(zip(images, titles)):
        axes[0, column].imshow(image, cmap="gray", vmin=0, vmax=255)
        axes[0, column].set_title(title)
        axes[0, column].axis("off")
        axes[1, column].hist(
            np.asarray(image).ravel(), bins=256, range=(0, 256), color="#315c9b"
        )
        axes[1, column].set_xlim(0, 255)
        axes[1, column].set_xlabel("Nilai piksel")
        axes[1, column].set_ylabel("Jumlah")
    plt.tight_layout()
    plt.show()

### ✏ Ubah — pilih kasus utama

Pada percobaan pertama, jangan ubah nilai. Setelah berhasil, satu-satunya parameter di sel ini yang boleh diganti adalah `CASE_ID`.

In [ ]:
CASE_ID = "test_cate1_001"

if "load_case" not in globals():
    raise RuntimeError("Jalankan sel setup dan helper terlebih dahulu.")
case = load_case(CASE_ID)
opg = case["image"]
print("Kasus siap:", case["case_id"])

## 3. OPG sebagai array

Sebuah citra digital dapat dibaca sebagai susunan angka. Untuk array RGB, `shape` ditulis `(tinggi, lebar, kanal)`. Karena itu tinggi berada sebelum lebar.

### ▶ Jalankan — inspeksi `shape`, `dtype`, dan rentang piksel

In [ ]:
if "opg" not in globals():
    raise RuntimeError("Kasus belum dimuat. Jalankan sel CASE_ID terlebih dahulu.")
print("shape :", opg.shape)
print("dtype :", opg.dtype)
print("minimum–maksimum:", int(opg.min()), "–", int(opg.max()))
print("piksel [y=400, x=800]:", opg[400, 800].tolist())

### 🩺 Diskusikan — membaca bentuk array

Pada aset ini, keluaran yang diharapkan adalah tinggi sekitar 942, lebar 2000, dan 3 kanal RGB. Ketiga kanal pada PNG ini membawa tampilan radiograf grayscale yang sama/nyaris sama.

- `uint8` berarti setiap kanal menyimpan bilangan bulat 0–255.
- Nilai piksel adalah intensitas citra, **bukan ukuran penyakit**.
- Urutan indeks piksel adalah `[y, x]`, setara dengan `[baris, kolom]`.

### ▶ Jalankan — tampilkan OPG dan histogram

In [ ]:
if "opg" not in globals():
    raise RuntimeError("Kasus belum dimuat. Jalankan sel CASE_ID terlebih dahulu.")
opg_gray = cv2.cvtColor(opg, cv2.COLOR_RGB2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(opg_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(CASE_ID)
axes[0].axis("off")
axes[1].hist(opg_gray.ravel(), bins=256, range=(0, 256), color="#315c9b")
axes[1].set(xlim=(0, 255), xlabel="Nilai piksel", ylabel="Jumlah piksel")
plt.show()

### 🩺 Diskusikan — histogram tidak tahu anatomi

Histogram menghitung berapa banyak piksel pada setiap tingkat intensitas, tetapi membuang lokasi spasial. Dua citra dengan histogram mirip dapat menunjukkan anatomi berbeda. Karena itu histogram membantu memahami kontras global, bukan menilai temuan sendirian.

### ✅ Checkpoint 2 dan istirahat (menit 75–90)

Sebelum istirahat, jawab berpasangan:

1. Mengapa `opg.shape[0]` adalah tinggi?
2. Apa beda `shape` dan `dtype`?
3. Mengapa nilai piksel tidak boleh dibaca sebagai tingkat keparahan?

Setelah istirahat, pastikan sel kasus masih menampilkan `Kasus siap: test_cate1_001`.

## 4. Bounding box, ROI, dan slicing

Kotak ditulis sebagai `[x1, y1, x2, y2]`:

- `(x1, y1)` = sudut kiri atas;
- `(x2, y2)` = sudut kanan bawah;
- lebar = `x2 - x1`; tinggi = `y2 - y1`.

Array dipotong dengan urutan **baris lalu kolom**, sehingga rumusnya:

```python
crop = image[y1:y2, x1:x2]
```

Nilai `y2` dan `x2` tidak ikut diambil. Ini disebut batas akhir eksklusif.

### ▶ Jalankan — baca anotasi publik

In [ ]:
if "case" not in globals():
    raise RuntimeError("Kasus belum dimuat. Jalankan sel CASE_ID terlebih dahulu.")
print("Jumlah kotak anotasi:", len(case["annotations"]))
for nomor, anotasi in enumerate(case["annotations"], start=1):
    box = [round(v) for v in anotasi["bbox_xyxy"]]
    print(nomor, anotasi["label_id"], box)

### ✏ Ubah — tentukan ROI

Mulai dengan nilai berikut. Setelah gambar tampil, ubah **satu koordinat saja**, jalankan ulang, dan amati arah pergeserannya.

In [ ]:
ROI = [620, 270, 920, 710]

if "show_roi" not in globals() or "opg" not in globals() or "case" not in globals():
    raise RuntimeError("Jalankan sel setup, helper, dan CASE_ID terlebih dahulu.")
roi_crop = show_roi(opg, ROI, case["annotations"])
print("Ukuran crop:", roi_crop.shape)

### ▶ Jalankan — buktikan hubungan koordinat dan slicing

Perhatikan bahwa ROI ditulis `x, y`, tetapi slicing ditulis `y, x`.

In [ ]:
if "ROI" not in globals() or "opg" not in globals() or "roi_crop" not in globals():
    raise RuntimeError("ROI belum siap. Jalankan sel ROI terlebih dahulu.")
x1, y1, x2, y2 = ROI
roi_manual = opg[y1:y2, x1:x2]

print("Slicing:", f"opg[{y1}:{y2}, {x1}:{x2}]")
print("Sama dengan hasil helper:", np.array_equal(roi_crop, roi_manual))

### 🩺 Diskusikan — kotak bukan kebenaran biologis

Bounding box adalah representasi anotator: kotak dapat mencakup jaringan sekitar, tidak menunjukkan batas struktur secara presisi, dan dapat berbeda antar-anotator. Crop juga menghilangkan konteks anatomi. Sebutkan satu situasi ketika konteks di luar ROI penting.

## 5. Equalization dan CLAHE

- **Histogram equalization** memetakan kontras menggunakan seluruh citra.
- **CLAHE** bekerja per area kecil dan membatasi penguatan kontras agar tidak berlebihan.

Agar perbandingan jujur, ketiga panel berikut memakai skala tampilan yang sama: 0–255.

### ✏ Ubah — kekuatan CLAHE

Jalankan dengan `2.0`, lalu coba `1.0` dan `4.0`. Ubah hanya `CLAHE_CLIP`.

In [ ]:
CLAHE_CLIP = 2.0

if "apply_clahe" not in globals() or "opg" not in globals():
    raise RuntimeError("Jalankan sel setup, helper, dan CASE_ID terlebih dahulu.")
opg_clahe = apply_clahe(opg, CLAHE_CLIP)
print("CLAHE selesai dengan clip limit:", CLAHE_CLIP)

### ▶ Jalankan — bandingkan citra dan histogram

In [ ]:
if "opg_gray" not in globals() or "opg_clahe" not in globals():
    raise RuntimeError("Jalankan sel histogram dan CLAHE terlebih dahulu.")
opg_equalized = cv2.equalizeHist(opg_gray)

_show_image_and_histogram(
    [opg_gray, opg_equalized, opg_clahe],
    ["Asli", "Equalization global", f"CLAHE (clip={CLAHE_CLIP})"],
)

### 🩺 Diskusikan — perubahan tampilan bukan informasi baru

Equalization dan CLAHE mengubah pemetaan intensitas agar pola tertentu lebih mudah terlihat. Keduanya:

- dapat memperkuat noise, artefak, atau batas yang semula samar;
- dapat membuat struktur terlihat lebih tegas tanpa menambah sinyal yang tidak direkam detektor;
- tidak memperbaiki positioning, gerakan, superimposisi, atau eksposur yang gagal; dan
- harus dievaluasi bersama citra asli, bukan menggantikannya.

Bandingkan area yang sama di ketiga panel. Mana yang menjadi lebih mudah dibaca? Adakah noise/artefak yang ikut menguat?

### PNG latihan bukan DICOM klinis

Contoh ini adalah **PNG 8-bit** dengan nilai 0–255. DICOM klinis dapat mempunyai bit depth lebih tinggi, rentang nilai berbeda, metadata akuisisi, rescale, photometric interpretation, serta windowing. Karena itu, kode pengolahan PNG di notebook ini tidak boleh langsung disalin ke alur DICOM klinis tanpa validasi format dan kebijakan institusi.

## 6. Tantangan kasus kedua (berpasangan)

### ✏ Ubah — `test_cate1_012`

Sebelum menjalankan, isi bagian kosong ini di kertas/chat pasangan:

```text
CASE_ID = "______________"
ROI = [____, ____, ____, ____]
crop = image[____:____, ____:____]
```

Tujuan: memilih kedua area anotasi pada kasus latihan, lalu menghubungkan ROI dengan urutan slicing.

In [ ]:
CASE_ID = "test_cate1_012"
ROI = [740, 500, 1230, 720]

if "load_case" not in globals() or "show_roi" not in globals():
    raise RuntimeError("Jalankan sel setup dan helper terlebih dahulu.")
case_latihan = load_case(CASE_ID)
opg_latihan = case_latihan["image"]
crop_latihan = show_roi(opg_latihan, ROI, case_latihan["annotations"])
print("Crop latihan:", crop_latihan.shape)

### ✅ Self-check otomatis

Sel ini tidak menilai interpretasi radiograf. Ia hanya memeriksa keadaan teknis: kasus benar, ROI tidak kosong, koordinat berada dalam citra, dan tipe data sesuai.

In [ ]:
if "opg_latihan" not in globals() or "crop_latihan" not in globals():
    raise RuntimeError("Jalankan sel tantangan kasus kedua terlebih dahulu.")
tinggi, lebar = opg_latihan.shape[:2]
x1, y1, x2, y2 = ROI

assert CASE_ID == "test_cate1_012", "Gunakan kasus latihan yang diminta."
assert 0 <= x1 < x2 <= lebar and 0 <= y1 < y2 <= tinggi, "ROI di luar citra."
assert crop_latihan.size > 0, "ROI menghasilkan crop kosong."
assert opg_latihan.dtype == np.uint8, "Aset seharusnya uint8."
print("Checkpoint teknis lulus.")

<details>
<summary><strong>✅ Buka solusi fill-in-the-blank setelah mencoba</strong></summary>

```python
CASE_ID = "test_cate1_012"
ROI = [740, 500, 1230, 720]
crop = image[500:720, 740:1230]
```

Mengapa urutan terakhir berbeda? Bounding box memakai `(x, y)`, sedangkan array memakai `[baris, kolom]` atau `[y, x]`.

Jawaban ini bukan satu-satunya ROI yang valid. ROI lain dapat dipakai selama berada dalam batas citra, tidak kosong, dan tujuan pengamatan dijelaskan.
</details>

## Glosarium mini

| Istilah | Arti ringkas |
|---|---|
| **array** | susunan angka dengan dimensi tertentu |
| **assignment** | menyimpan nilai dengan `=` |
| **indeks** | posisi elemen; dimulai dari 0 |
| **shape** | ukuran array, misalnya `(tinggi, lebar, kanal)` |
| **dtype** | tipe data elemen, misalnya `uint8` |
| **piksel** | elemen terkecil pada citra raster |
| **bounding box** | kotak `[x1, y1, x2, y2]` |
| **ROI** | *region of interest*, area yang dipilih untuk diamati |
| **slicing** | mengambil bagian array, misalnya `[y1:y2, x1:x2]` |
| **histogram** | hitungan piksel menurut intensitas |
| **CLAHE** | equalization lokal dengan pembatasan penguatan kontras |
| **checksum SHA-256** | sidik jari digital untuk memeriksa integritas berkas |

## Exit ticket — jawab empat butir

1. Tulis satu baris Python untuk mengambil ROI `[x1, y1, x2, y2]` dari array `image`.
2. Apa arti `shape = (942, 2000, 3)`?
3. Sebutkan satu manfaat dan satu risiko CLAHE.
4. Mengapa radiograf pasien tidak boleh diunggah ke Colab untuk mengganti aset latihan?

**Bawa ke Sesi 2:** satu contoh bagaimana pilihan ROI atau peningkatan kontras dapat memengaruhi apa yang manusia atau sistem AI perhatikan.

---

## Sumber dan batas interpretasi

- Dataset publik: [Children's Dental Panoramic Radiographs Dataset](https://doi.org/10.6084/m9.figshare.c.6317013.v1).
- Konteks dataset: [Scientific Data 10, 380 (2023)](https://doi.org/10.1038/s41597-023-02237-5).
- Anotasi adalah label dataset publik dan dapat mengandung ketidakpastian atau perbedaan antar-anotator.
- Subset pediatrik, format PNG, dan konteks sumber membatasi generalisasi ke populasi, perangkat, dan protokol lain.

Notebook tidak mengunggah data secara otomatis. Unduhan hanya mengambil aset publik yang namanya tercantum di manifest, dan setiap citra diperiksa dengan SHA-256 sebelum dipakai.